# Double Demeaning Analysis Example: Smoking and Birth Weight

This notebook demonstrates the double demeaning technique using the infant birth weight dataset from Abrevaya (2006), replicating the analysis from Giesselmann & Schmidt-Catran (2022).

## Research Question
Does the effect of maternal age on birth weight vary by smoking status?

## Dataset
- **Source**: Abrevaya (2006) infant birth weight study
- **Units**: Mothers (momid)
- **Time**: Birth order (idx)
- **Outcome**: Birth weight in grams (birwt)
- **Key Variables**: Maternal age (mage), Smoking status (smoke)
- **Controls**: Marital status (married)


In [1]:
import logging

import numpy as np
import pandas as pd

from dd_ie import DoubleDemeanAnalysis

# Configure logging so verbose output is visible in the notebook
logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")

print("dd_ie imported successfully.")


dd_ie imported successfully.


## Step 1: Load Data

**Important**: You need to manually download the smoking.dta file from:
https://www.stata-press.com/data/mlmus2/smoking.dta

Place it in the same folder as this notebook.


In [3]:
# Load the smoking dataset
try:
    df = pd.read_stata('smoking.dta')
    print(f"✅ SUCCESS: Loaded smoking dataset with {len(df):,} observations")
    print(f"   Panel structure: {df['momid'].nunique():,} mothers, {df['idx'].nunique()} time periods")
except FileNotFoundError:
    print("❌ ERROR: smoking.dta not found!")
    print("")
    print("Please download manually from:")
    print("https://www.stata-press.com/data/mlmus2/smoking.dta")
    print("")
    print("Save the file in the same folder as this notebook and re-run this cell.")
    raise

# Display basic information
print(f"\n📊 Dataset Overview:")
print(f"   Shape: {df.shape}")
print(f"   Variables: {list(df.columns)[:10]}...")  # Show first 10 columns

# Show first few rows
print(f"\n📋 First 5 rows:")
df.head()


✅ SUCCESS: Loaded smoking dataset with 8,604 observations
   Panel structure: 3,978 mothers, 3 time periods

📊 Dataset Overview:
   Shape: (8604, 24)
   Variables: ['momid', 'idx', 'stateres', 'mage', 'meduc', 'mplbir', 'nlbnl', 'gestat', 'birwt', 'cigs']...

📋 First 5 rows:


,momid,idx,stateres,mage,meduc,mplbir,nlbnl,gestat,birwt,cigs,...,hsgrad,somecoll,collgrad,magesq,black,kessner2,kessner3,novisit,pretri2,pretri3
0,14.0,1.0,AL,16.0,10.0,VA,0.0,24.0,2790.0,0.0,...,0.0,0.0,0.0,256.0,Black,0.0,1.0,0.0,0.0,1.0
1,14.0,2.0,AL,17.0,10.0,VA,1.0,42.0,2693.0,0.0,...,0.0,0.0,0.0,289.0,Black,0.0,1.0,0.0,0.0,1.0
2,14.0,3.0,AL,20.0,10.0,VA,2.0,39.0,3600.0,0.0,...,0.0,0.0,0.0,400.0,Black,0.0,0.0,0.0,0.0,0.0
3,25.0,1.0,AL,23.0,11.0,NJ,2.0,41.0,2778.0,0.0,...,0.0,0.0,0.0,529.0,Black,1.0,0.0,0.0,1.0,0.0
4,25.0,2.0,AL,24.0,11.0,NJ,3.0,37.0,2835.0,0.0,...,0.0,0.0,0.0,576.0,Black,1.0,0.0,0.0,1.0,0.0


## Step 2: Initialize Analysis

We'll analyze how the effect of maternal age (mage) on birth weight (birwt) varies by smoking status (smoke).


In [4]:
# Initialize the analysis
analysis = DoubleDemeanAnalysis(
    data=df,
    unit_var='momid',      # Mother ID (unit identifier)
    time_var='idx',        # Birth order (time identifier)
    y_var='birwt',         # Birth weight in grams (outcome)
    x_var='mage',          # Maternal age (first interacting variable)
    z_var='smoke',         # Smoking status (second interacting variable)
    w_vars=['married']     # Marital status (control variable)
)

print("✅ Analysis initialized successfully!")
print("   Ready to run double demeaning analysis...")


INFO - Panel Data Summary:
INFO -   Total observations: 8604
INFO -   Number of units: 3978
INFO -   Time periods per unit: 2-3
INFO -   Panel type: Unbalanced
INFO - Data prepared: 8604 observations, 3978 units


✅ Analysis initialized successfully!
   Ready to run double demeaning analysis...


## Step 3: Run Complete Analysis

This will perform:
1. Data validation and preparation
2. Grand mean centering  
3. Double demeaning implementation
4. Model estimation (Standard FE vs Double Demeaned FE)
5. Hausman test for systematic differences


In [7]:
# Run the complete analysis
results = analysis.run_analysis(
    center_variables=True,    # Apply grand mean centering (matches Stata)
    run_hausman=True,        # Perform Hausman test
    verbose=True             # Show detailed output
)

print("\n" + "="*80)
print("🎉 ANALYSIS COMPLETED SUCCESSFULLY!")
print("="*80)


INFO - STARTING DOUBLE DEMEANING ANALYSIS
INFO - STARTING DOUBLE DEMEANING ANALYSIS
INFO - Dataset: 8604 observations, 22 variables
INFO - Dataset: 8604 observations, 22 variables
INFO - Panel structure: momid (units) x idx (time)
INFO - Panel structure: momid (units) x idx (time)
INFO - Analysis: birwt ~ mage x smoke + controls
INFO - Analysis: birwt ~ mage x smoke + controls
INFO - Step 1: Data Preparation
INFO - Step 1: Data Preparation
INFO -   Panel: 3978 units, 2-3 periods per unit (mean 2.2)
INFO -   Panel: 3978 units, 2-3 periods per unit (mean 2.2)
WARNING - 3330 units have <= 2 periods. Double demeaning requires T > 2. Consider filtering.
WARNING - 3330 units have <= 2 periods. Double demeaning requires T > 2. Consider filtering.
INFO - Step 2: Grand Mean Centering
INFO - Step 2: Grand Mean Centering
INFO -   birwt: mean before = 3469.93115, mean after = -0.0000740026
INFO -   birwt: mean before = 3469.93115, mean after = -0.0000740026
INFO -   mage: mean before = 28.59182, m


🎉 ANALYSIS COMPLETED SUCCESSFULLY!


## Step 4: Interpret Results

Let's examine the key findings and create visualizations.


In [8]:
# Access results via the typed dataclass API
comparison = results.comparison
hausman = results.hausman

print("RESULTS SUMMARY")
print("=" * 40)

print("\nCoefficient Comparison:")
print(comparison.table.to_string(index=False, float_format="%.4f"))
print(f"\nInteraction coefficient difference: {comparison.interaction_difference:.4f}")

if hausman is not None:
    print(f"\nHausman Test:")
    print(f"  Chi-square:  {hausman.statistic:.2f}")
    print(f"  P-value:     {hausman.p_value:.4f}")
    print(f"  Conclusion:  {hausman.conclusion}")

    if hausman.p_value >= 0.05:
        print("\n  No evidence of systematic bias.")
        print("  Both estimators appear consistent. Standard FE is more efficient.")
    else:
        print("\n  Evidence of systematic bias detected!")
        print("  Standard FE may be inconsistent. Prefer double-demeaned estimator.")


RESULTS SUMMARY

Coefficient Comparison:
      Variable  Std_FE_Coef  Std_FE_SE   DD_Coef   DD_SE  Difference
          mage      23.2820     3.1385   23.1245  3.1253      0.1575
         smoke     -97.1669    34.7692 -102.5927 31.8109      5.4258
int_mage_smoke       3.7946     5.1575  -78.9843 56.6387     82.7789

Interaction coefficient difference: 82.7789

Hausman Test:
  Chi-square:  3.00
  P-value:     0.3922
  Conclusion:  NO_SYSTEMATIC_BIAS

  No evidence of systematic bias.
  Both estimators appear consistent. Standard FE is more efficient.
